# M19c — Test-time prediction-variation indicators

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Test whether label-free prediction variation across the validation-safe temperature region is associated with the labelled improvement that was retrospectively available inside that region.

**Provenance.** Historical manuscript values are preserved exactly. The original run-level notebook is unavailable, so the executable analysis below is an independent replication using the later preserved task generator and explicit window definitions.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
SEED=20260623
TRIALS=12
TASKS=["controlled_d20_white_plus_distractor","memory_d10","narma10","lorenz_x"]
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=0.8,ridge=1e-5)
hist=pd.read_csv(FROZEN/"historical_reference_metrics.csv")
display(hist[hist.notebook_id=="M19c"])

,notebook_id,metric,value,ci_low,ci_high,provenance
10,M19c,safe_dispersion_gain_corr,0.41200,NaN,NaN,historical manuscript result
11,M19c,anchor_spread_gain_corr,0.40500,NaN,NaN,historical manuscript result
12,M19c,safe_dispersion_high_minus_low_gain,0.01689,0.01227,0.02196,historical manuscript result
13,M19c,anchor_spread_high_minus_low_gain,0.01702,0.01243,0.02207,historical manuscript result
14,M19c,composite_high_minus_low_gain,0.01697,0.01228,0.02197,historical manuscript result


## Window-level replication

In [3]:
parts=[]
for task in TASKS:
    for trial in range(TRIALS):
        case=tcr.evaluate_case(task,trial,"temperature",SEED,return_predictions=True,**CONFIG)
        w=tcr.window_rows(case,window_size=50,stride=25)
        w["task"]=task; w["trial"]=trial; w["case_id"]=f"{task}__{trial}"
        parts.append(w)
windows=pd.concat(parts,ignore_index=True)
windows.to_csv(REPRO/"m19c_replication_window_metrics.csv",index=False)

def quartile_difference(frame, indicator):
    work=frame[np.isfinite(frame[indicator])].copy()
    q1,q3=work[indicator].quantile([.25,.75])
    low=work[work[indicator]<=q1].safe_gain
    high=work[work[indicator]>=q3].safe_gain
    return float(high.mean()-low.mean())

summary=pd.DataFrame([{
    "n_windows":len(windows),
    "safe_dispersion_gain_corr":tcr.safe_corr(windows.safe_dispersion,windows.safe_gain),
    "anchor_spread_gain_corr":tcr.safe_corr(windows.anchor_spread,windows.safe_gain),
    "local_sensitivity_gain_corr":tcr.safe_corr(windows.local_sensitivity,windows.safe_gain),
    "safe_dispersion_high_minus_low_gain":quartile_difference(windows,"safe_dispersion"),
    "anchor_spread_high_minus_low_gain":quartile_difference(windows,"anchor_spread"),
}])
summary.to_csv(REPRO/"m19c_replication_summary.csv",index=False)
display(summary.round(6))

,n_windows,safe_dispersion_gain_corr,anchor_spread_gain_corr,local_sensitivity_gain_corr,safe_dispersion_high_minus_low_gain,anchor_spread_high_minus_low_gain
0,912,0.322929,0.387012,0.327759,0.017544,0.020626


The manuscript's primary indicator claim remains the historical frozen result. This replication checks the direction and interpretation using a separately reconstructed panel.